# Atividade 5 - Evolucao do IDHM por UF

Analise da tabela `Tabela4.csv` e graficos com `matplotlib`.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt

## 1. Abrir o CSV e limpar colunas vazias

In [ ]:
df = pd.read_csv("Tabela4.csv", sep=";", decimal=",", skiprows=1, encoding="latin1")

# Remove colunas vazias criadas por separadores extras no CSV.
df = df.dropna(axis=1, how="all")
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

df[anos] = df[anos].apply(pd.to_numeric, errors="coerce")
df.head()

## 2. Ordenar os estados por maior IDH em 2024

In [ ]:
ranking_2024 = df[["Sigla", "Estado", "2024"]].sort_values("2024", ascending=False)
ranking_2024

## 3. Maior melhora entre 1991 e 2024

In [ ]:
df["Melhoria_1991_2024"] = df["2024"] - df["1991"]
maior_melhoria = df.loc[df["Melhoria_1991_2024"].idxmax()]

print(f"Maior melhoria: {maior_melhoria['Estado']} ({maior_melhoria['Sigla']})")
print(f"Variacao: {maior_melhoria['Melhoria_1991_2024']:.3f}")

## 4. Verificar se algum estado piorou

In [ ]:
pioraram = df.loc[df["2024"] < df["1991"], ["Sigla", "Estado", "1991", "2024"]]

if pioraram.empty:
    print("Nenhum estado piorou de 1991 para 2024.")
else:
    pioraram

## 5. Passar de formato largo para formato longo com melt

In [ ]:
id_vars = [c for c in df.columns if c not in anos]

df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)

df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

## 6. Plotar apenas Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2)
ax.set_title("Evolucao do IDH - Minas Gerais")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 7. Plotar a evolucao do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolucao do IDH por estado (1991-2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()